# 🔥 PyTorch: Regularization, Generalization & Data Augmentation
## A Comprehensive A/B Testing Guide

**Assignment Part 1 — PyTorch Edition**

This notebook mirrors the TensorFlow edition, implementing all regularization techniques and data augmentation strategies in PyTorch. Each technique includes **A/B comparisons** with clear metrics and visualizations.

### Table of Contents
1. **Setup & Dataset Preparation**
2. **L1 & L2 Regularization (Weight Decay)**
3. **Dropout Regularization**
4. **Early Stopping**
5. **Monte Carlo Dropout**
6. **Weight Initialization Strategies**
7. **Batch Normalization**
8. **Custom Dropout & Custom Regularization**
9. **TensorBoard Integration with PyTorch**
10. **Optuna Hyperparameter Optimization** (PyTorch equivalent of Keras Tuner)
11. **torchvision Data Augmentation**
12. **Multi-Domain Data Augmentation** (Image, Text, Time Series, Tabular, Audio)

---


## 1. Setup & Dataset Preparation

In [ ]:
# ============================================================
# Install dependencies
# ============================================================
!pip install -q optuna nlpaug audiomentations tsaug albumentations tensorboard

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import copy
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [ ]:
# ============================================================
# Load CIFAR-10
# ============================================================
transform_basic = T.Compose([
    T.ToTensor(),
    # Note: ToTensor already normalizes to [0,1]
])

train_dataset_full = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_basic)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_basic)

# Split train into train + validation
train_size = 40000
val_size = 10000
train_dataset, val_dataset = random_split(
    train_dataset_full, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")

# Preview
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=10)
    ax.axis('off')
plt.suptitle("CIFAR-10 Sample Images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Helper: Configurable CNN
# ============================================================
class FlexibleCNN(nn.Module):
    """
    Configurable CNN for A/B testing different regularization strategies.
    """
    def __init__(self, num_classes=10, use_dropout=False, dropout_rate=0.5,
                 use_batchnorm=False, initializer='kaiming'):
        super().__init__()
        self.use_dropout = use_dropout
        self.use_batchnorm = use_batchnorm
        self.dropout_rate = dropout_rate

        # Block 1
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) if use_batchnorm else nn.Identity()
        self.bn2 = nn.BatchNorm2d(32) if use_batchnorm else nn.Identity()
        self.pool1 = nn.MaxPool2d(2)
        self.drop1 = nn.Dropout(dropout_rate) if use_dropout else nn.Identity()

        # Block 2
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64) if use_batchnorm else nn.Identity()
        self.bn4 = nn.BatchNorm2d(64) if use_batchnorm else nn.Identity()
        self.pool2 = nn.MaxPool2d(2)
        self.drop2 = nn.Dropout(dropout_rate) if use_dropout else nn.Identity()

        # Classifier
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.bn_fc = nn.BatchNorm1d(128) if use_batchnorm else nn.Identity()
        self.drop_fc = nn.Dropout(dropout_rate) if use_dropout else nn.Identity()
        self.fc2 = nn.Linear(128, num_classes)

        # Apply initialization
        self._init_weights(initializer)

    def _init_weights(self, method):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                if method == 'kaiming' or method == 'he':
                    nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                elif method == 'xavier' or method == 'glorot':
                    nn.init.xavier_normal_(m.weight)
                elif method == 'orthogonal':
                    nn.init.orthogonal_(m.weight)
                elif method == 'lecun':
                    nn.init.kaiming_normal_(m.weight, nonlinearity='linear')
                elif method == 'zeros':
                    nn.init.zeros_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.drop1(self.pool1(x))

        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.drop2(self.pool2(x))

        x = x.view(x.size(0), -1)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.drop_fc(x)
        x = self.fc2(x)
        return x


def train_model(model, epochs=30, lr=1e-3, weight_decay=0.0,
                l1_lambda=0.0, extra_callbacks=None):
    """
    Train a PyTorch model and return history dict.
    Supports L2 via weight_decay and explicit L1 penalty.
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

    for epoch in range(epochs):
        # Training
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Explicit L1 regularization
            if l1_lambda > 0:
                l1_norm = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_norm

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total

        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)

        # Callback support (e.g., early stopping)
        if extra_callbacks:
            for cb in extra_callbacks:
                stop = cb(epoch, history, model)
                if stop:
                    return history

    return history


class HistoryWrapper:
    """Wrap dict to mimic Keras history for plot_ab compatibility."""
    def __init__(self, d):
        self.history = d


def plot_ab(hist_a, hist_b, label_a="Baseline", label_b="Regularized"):
    """Side-by-side loss & accuracy comparison."""
    ha = hist_a.history if hasattr(hist_a, 'history') else hist_a
    hb = hist_b.history if hasattr(hist_b, 'history') else hist_b

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(ha['loss'], label=f'{label_a} train', linestyle='--')
    axes[0].plot(ha['val_loss'], label=f'{label_a} val')
    axes[0].plot(hb['loss'], label=f'{label_b} train', linestyle='--')
    axes[0].plot(hb['val_loss'], label=f'{label_b} val')
    axes[0].set_title("Loss Comparison", fontweight='bold')
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(ha['accuracy'], label=f'{label_a} train', linestyle='--')
    axes[1].plot(ha['val_accuracy'], label=f'{label_a} val')
    axes[1].plot(hb['accuracy'], label=f'{label_b} train', linestyle='--')
    axes[1].plot(hb['val_accuracy'], label=f'{label_b} val')
    axes[1].set_title("Accuracy Comparison", fontweight='bold')
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    val_loss_a = min(ha['val_loss'])
    val_loss_b = min(hb['val_loss'])
    val_acc_a = max(ha['val_accuracy'])
    val_acc_b = max(hb['val_accuracy'])
    print(f"\n{'Metric':<25} {label_a:<15} {label_b:<15}")
    print("-" * 55)
    print(f"{'Best Val Loss':<25} {val_loss_a:<15.4f} {val_loss_b:<15.4f}")
    print(f"{'Best Val Accuracy':<25} {val_acc_a:<15.4f} {val_acc_b:<15.4f}")


## 2. L1 & L2 Regularization (Weight Decay)

In PyTorch, L2 regularization is commonly applied via the `weight_decay` parameter in optimizers. L1 must be added explicitly to the loss.

**A/B Test:** Baseline vs. L1, L2, and Elastic Net.


In [ ]:
# ============================================================
# 2a — Baseline (no regularization)
# ============================================================
print("Training BASELINE model (no regularization)...")
baseline_model = FlexibleCNN()
hist_baseline = train_model(baseline_model, epochs=30)
hist_baseline = HistoryWrapper(hist_baseline)


In [ ]:
# ============================================================
# 2b — L2 Regularization (weight_decay)
# ============================================================
print("Training L2 REGULARIZED model (weight_decay=1e-4)...")
l2_model = FlexibleCNN()
hist_l2 = train_model(l2_model, epochs=30, weight_decay=1e-4)
hist_l2 = HistoryWrapper(hist_l2)

plot_ab(hist_baseline, hist_l2, "Baseline", "L2 (wd=1e-4)")


In [ ]:
# ============================================================
# 2c — L1 Regularization (explicit penalty)
# ============================================================
print("Training L1 REGULARIZED model (lambda=1e-5)...")
l1_model = FlexibleCNN()
hist_l1 = train_model(l1_model, epochs=30, l1_lambda=1e-5)
hist_l1 = HistoryWrapper(hist_l1)

plot_ab(hist_baseline, hist_l1, "Baseline", "L1 (λ=1e-5)")


In [ ]:
# ============================================================
# 2d — Elastic Net (L1 + L2)
# ============================================================
print("Training ELASTIC NET model (L1=1e-5 + L2=1e-4)...")
elastic_model = FlexibleCNN()
hist_elastic = train_model(elastic_model, epochs=30, weight_decay=1e-4, l1_lambda=1e-5)
hist_elastic = HistoryWrapper(hist_elastic)

plot_ab(hist_baseline, hist_elastic, "Baseline", "ElasticNet")


In [ ]:
# ============================================================
# 2e — Weight distribution analysis
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
models_list = [baseline_model, l1_model, l2_model, elastic_model]
titles = ["No Reg", "L1", "L2", "Elastic Net"]

for ax, m, title in zip(axes, models_list, titles):
    all_weights = torch.cat([p.data.cpu().flatten()
                             for p in m.parameters()
                             if p.dim() >= 2]).numpy()
    ax.hist(all_weights, bins=100, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.3)
    ax.set_title(f"{title}\nstd={all_weights.std():.4f}", fontweight='bold')
    ax.set_xlabel("Weight Value")
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)

plt.suptitle("Weight Distribution Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nL1 pushes weights toward exact zero (sparsity).")
print("L2 shrinks all weights uniformly. Elastic Net combines both effects.")


## 3. Dropout Regularization

**A/B Test:** Baseline vs. Dropout, plus a sweep of dropout rates.


In [ ]:
# ============================================================
# 3a — Dropout A/B test
# ============================================================
print("Training model WITH DROPOUT (rate=0.3)...")
dropout_model = FlexibleCNN(use_dropout=True, dropout_rate=0.3)
hist_dropout = train_model(dropout_model, epochs=30)
hist_dropout = HistoryWrapper(hist_dropout)

plot_ab(hist_baseline, hist_dropout, "No Dropout", "Dropout 0.3")


In [ ]:
# ============================================================
# 3b — Dropout rate sweep
# ============================================================
dropout_rates = [0.1, 0.3, 0.5, 0.7]
dropout_histories = {}

for rate in dropout_rates:
    print(f"Training dropout rate = {rate}...")
    m = FlexibleCNN(use_dropout=True, dropout_rate=rate)
    h = train_model(m, epochs=25)
    dropout_histories[rate] = h

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for rate, h in dropout_histories.items():
    axes[0].plot(h['val_loss'], label=f'rate={rate}')
    axes[1].plot(h['val_accuracy'], label=f'rate={rate}')

axes[0].set_title("Val Loss by Dropout Rate", fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title("Val Accuracy by Dropout Rate", fontweight='bold')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val Accuracy")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_rate = min(dropout_histories, key=lambda r: min(dropout_histories[r]['val_loss']))
print(f"\nBest dropout rate by val loss: {best_rate}")


## 4. Early Stopping

PyTorch doesn't have built-in early stopping like Keras, so we implement it as a callback function.


In [ ]:
# ============================================================
# 4a — Early stopping implementation
# ============================================================
class EarlyStopper:
    """
    Monitors validation loss and stops training when it plateaus.
    Restores best model weights.
    """
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_model_state = None

    def __call__(self, epoch, history, model):
        val_loss = history['val_loss'][-1]
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print(f"  ✓ Early stopping at epoch {epoch + 1} (patience={self.patience})")
                if self.best_model_state:
                    model.load_state_dict(self.best_model_state)
                return True  # Signal to stop
        return False

# Without early stopping (long training)
print("Training 60 epochs WITHOUT early stopping...")
long_model = FlexibleCNN()
hist_long = train_model(long_model, epochs=60)
hist_long = HistoryWrapper(hist_long)

# With early stopping
print("\nTraining WITH early stopping (patience=7)...")
es_model = FlexibleCNN()
stopper = EarlyStopper(patience=7)
hist_es = train_model(es_model, epochs=60, extra_callbacks=[stopper])
hist_es = HistoryWrapper(hist_es)

plot_ab(hist_long, hist_es, "No Early Stop (60ep)", "Early Stopping")
actual_epochs = len(hist_es.history['loss'])
print(f"\nEarly stopping halted at epoch: {actual_epochs}")
print(f"Saved {60 - actual_epochs} epochs of unnecessary training!")


## 5. Monte Carlo Dropout

In PyTorch, we keep dropout active at inference by calling `model.train()` or using a custom forward mode. We collect multiple stochastic predictions and measure uncertainty.


In [ ]:
# ============================================================
# 5a — MC Dropout inference
# ============================================================
mc_model = FlexibleCNN(use_dropout=True, dropout_rate=0.3).to(device)
_ = train_model(mc_model, epochs=25)

NUM_MC_SAMPLES = 50

# Get test batch
test_images, test_labels = [], []
for imgs, lbls in test_loader:
    test_images.append(imgs)
    test_labels.append(lbls)
    if len(test_labels) * 256 >= 200:
        break
test_images = torch.cat(test_images)[:200].to(device)
test_labels = torch.cat(test_labels)[:200].numpy()

# MC Dropout: run forward passes with dropout ON
mc_model.train()  # Keep dropout active
mc_predictions = []
with torch.no_grad():
    for _ in range(NUM_MC_SAMPLES):
        # Manually enable dropout but disable batchnorm updates
        logits = mc_model(test_images)
        probs = F.softmax(logits, dim=1)
        mc_predictions.append(probs.cpu().numpy())

mc_predictions = np.stack(mc_predictions)  # (50, 200, 10)

# Aggregate
mean_probs = mc_predictions.mean(axis=0)
predictive_entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-10), axis=1)
predicted_classes = mean_probs.argmax(axis=1)

# Standard inference
mc_model.eval()
with torch.no_grad():
    standard_preds = mc_model(test_images).argmax(1).cpu().numpy()

print(f"MC Dropout accuracy:  {(predicted_classes == test_labels).mean():.4f}")
print(f"Standard accuracy:    {(standard_preds == test_labels).mean():.4f}")


In [ ]:
# ============================================================
# 5b — Uncertainty visualization
# ============================================================
correct_mask = predicted_classes == test_labels
incorrect_mask = ~correct_mask

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(predictive_entropy[correct_mask], bins=30, alpha=0.7,
        label='Correct', color='green', edgecolor='black', linewidth=0.3)
ax.hist(predictive_entropy[incorrect_mask], bins=30, alpha=0.7,
        label='Incorrect', color='red', edgecolor='black', linewidth=0.3)
ax.set_title("MC Dropout: Predictive Entropy Distribution", fontweight='bold')
ax.set_xlabel("Entropy (Uncertainty)")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Per-class uncertainty
print("\nMean entropy by prediction correctness:")
print(f"  Correct predictions:   {predictive_entropy[correct_mask].mean():.4f}")
print(f"  Incorrect predictions: {predictive_entropy[incorrect_mask].mean():.4f}")
print("\nHigher entropy → less confident → more likely wrong. MC Dropout detects unreliable predictions!")


## 6. Weight Initialization Strategies

| Initializer | Best For | PyTorch Function |
|---|---|---|
| **Kaiming (He) Normal** | ReLU, Leaky ReLU | `nn.init.kaiming_normal_` |
| **Xavier (Glorot) Normal** | Sigmoid, Tanh | `nn.init.xavier_normal_` |
| **Orthogonal** | RNNs, deep networks | `nn.init.orthogonal_` |
| **LeCun Normal** | SELU activations | `nn.init.kaiming_normal_(nonlinearity='linear')` |
| **Zeros** | ❌ Never for weights | `nn.init.zeros_` |


In [ ]:
# ============================================================
# 6a — Initializer comparison
# ============================================================
initializers = ['kaiming', 'xavier', 'orthogonal', 'lecun']
init_histories = {}

for name in initializers:
    print(f"Training with {name} initialization...")
    m = FlexibleCNN(initializer=name)
    h = train_model(m, epochs=20)
    init_histories[name] = h

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in init_histories.items():
    axes[0].plot(h['val_loss'], label=name)
    axes[1].plot(h['val_accuracy'], label=name)

axes[0].set_title("Val Loss by Initializer", fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val Loss")
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].set_title("Val Accuracy by Initializer", fontweight='bold')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val Accuracy")
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n{'Initializer':<20} {'Best Val Loss':<15} {'Best Val Acc':<15}")
print("-" * 50)
for name, h in init_histories.items():
    print(f"{name:<20} {min(h['val_loss']):<15.4f} {max(h['val_accuracy']):<15.4f}")


In [ ]:
# ============================================================
# 6b — Zeros initialization (anti-pattern)
# ============================================================
print("WARNING: Training with ALL ZEROS initialization...\n")
zero_model = FlexibleCNN(initializer='zeros')
hist_zero = train_model(zero_model, epochs=15)

print(f"\nFinal val accuracy with zeros: {hist_zero['val_accuracy'][-1]:.4f}")
print("Approximately random chance (10%). Symmetry prevents meaningful learning.")
print("Rule: ALWAYS use Kaiming (for ReLU) or Xavier (for sigmoid/tanh)!")


## 7. Batch Normalization

**A/B Test:** Standard CNN vs. BatchNorm-equipped CNN, including higher learning rate tolerance.


In [ ]:
# ============================================================
# 7a — BatchNorm A/B test
# ============================================================
print("Training model WITH Batch Normalization...")
bn_model = FlexibleCNN(use_batchnorm=True)
hist_bn = train_model(bn_model, epochs=30)
hist_bn = HistoryWrapper(hist_bn)

plot_ab(hist_baseline, hist_bn, "No BatchNorm", "With BatchNorm")


In [ ]:
# ============================================================
# 7b — BatchNorm with aggressive learning rate
# ============================================================
print("Training BatchNorm model at lr=5e-3...")
bn_fast = FlexibleCNN(use_batchnorm=True)
hist_bn_fast = train_model(bn_fast, epochs=30, lr=5e-3)
hist_bn_fast = HistoryWrapper(hist_bn_fast)

print("Training baseline at lr=5e-3...")
base_fast = FlexibleCNN()
hist_base_fast = train_model(base_fast, epochs=30, lr=5e-3)
hist_base_fast = HistoryWrapper(hist_base_fast)

plot_ab(hist_base_fast, hist_bn_fast, "No BN (lr=5e-3)", "BN (lr=5e-3)")
print("\nBatchNorm enables stable training with larger learning rates.")


## 8. Custom Dropout & Custom Regularization

Implementing custom regularization in PyTorch by subclassing `nn.Module`.


In [ ]:
# ============================================================
# 8a — Alpha Dropout (for SELU networks)
# ============================================================
class AlphaDropout(nn.Module):
    """
    Alpha Dropout for Self-Normalizing Neural Networks (SELU).
    Replaces dropped activations with the SELU saturation value
    instead of zero, preserving the self-normalizing property.
    """
    def __init__(self, rate=0.1):
        super().__init__()
        self.rate = rate
        self.alpha = 1.6732632423543772
        self.scale = 1.0507009873554805

    def forward(self, x):
        if not self.training:
            return x

        mask = (torch.rand_like(x) >= self.rate).float()
        saturation = -self.alpha * self.scale

        output = x * mask + saturation * (1.0 - mask)

        # Affine correction
        a = ((1.0 - self.rate) * (1.0 + self.rate * saturation**2)) ** (-0.5)
        b = -a * saturation * self.rate

        return a * output + b

# Test
alpha_drop = AlphaDropout(rate=0.2)
test_input = torch.randn(4, 8)
alpha_drop.train()
print("Alpha Dropout (training):", alpha_drop(test_input)[0, :4])
alpha_drop.eval()
print("Alpha Dropout (eval):    ", alpha_drop(test_input)[0, :4])


In [ ]:
# ============================================================
# 8b — Concrete Dropout (learnable rate)
# ============================================================
class ConcreteDropout(nn.Module):
    """
    Concrete Dropout: learns the optimal dropout probability during training
    using the Gumbel-Softmax relaxation.
    """
    def __init__(self, temperature=0.1, init_rate=0.27):
        super().__init__()
        self.temperature = temperature
        # Initialize logit so sigmoid(logit) ≈ init_rate
        init_logit = np.log(init_rate / (1 - init_rate))
        self.p_logit = nn.Parameter(torch.tensor(init_logit, dtype=torch.float32))

    def forward(self, x):
        if not self.training:
            return x

        p = torch.sigmoid(self.p_logit)
        u = torch.rand_like(x).clamp(1e-6, 1 - 1e-6)
        z = torch.sigmoid((torch.log(u) - torch.log(1 - u) + self.p_logit) / self.temperature)
        mask = 1.0 - z
        return x * mask / (1 - p + 1e-6)

    @property
    def dropout_rate(self):
        return torch.sigmoid(self.p_logit).item()

concrete = ConcreteDropout()
print(f"Initial learned dropout rate: {concrete.dropout_rate:.4f}")


In [ ]:
# ============================================================
# 8c — Custom Orthogonal Regularizer
# ============================================================
class OrthogonalRegularizer:
    """
    Adds an orthogonality penalty to a model's loss:
      penalty = strength * ||W^T W - I||_F
    Encourages weight matrices to preserve gradient norms.
    """
    def __init__(self, model, strength=1e-3):
        self.model = model
        self.strength = strength

    def penalty(self):
        total = 0.0
        for p in self.model.parameters():
            if p.dim() >= 2:
                w = p.view(-1, p.size(-1))
                product = w.T @ w
                identity = torch.eye(product.size(0), device=p.device)
                total += torch.sum((product - identity) ** 2)
        return self.strength * total


# A/B test with orthogonal regularizer
print("Training with ORTHOGONAL regularizer...")
ortho_model = FlexibleCNN().to(device)
ortho_reg = OrthogonalRegularizer(ortho_model, strength=1e-4)
optimizer = optim.Adam(ortho_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

ortho_history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

for epoch in range(25):
    ortho_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = ortho_model(images)
        loss = criterion(outputs, labels) + ortho_reg.penalty()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    ortho_history['loss'].append(running_loss / total)
    ortho_history['accuracy'].append(correct / total)

    ortho_model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = ortho_model(images)
            vl += criterion(outputs, labels).item() * images.size(0)
            _, predicted = outputs.max(1)
            vt += labels.size(0)
            vc += predicted.eq(labels).sum().item()
    ortho_history['val_loss'].append(vl / vt)
    ortho_history['val_accuracy'].append(vc / vt)

plot_ab(hist_baseline, HistoryWrapper(ortho_history), "Baseline", "Orthogonal Reg")


## 9. TensorBoard Integration with PyTorch

PyTorch integrates with TensorBoard via `torch.utils.tensorboard.SummaryWriter`.


In [ ]:
# ============================================================
# 9a — TensorBoard logging setup
# ============================================================
from torch.utils.tensorboard import SummaryWriter
import datetime

log_dir = f"runs/pytorch_experiment_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir)

tb_model = FlexibleCNN(use_batchnorm=True, use_dropout=True, dropout_rate=0.3).to(device)
optimizer = optim.Adam(tb_model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)
criterion = nn.CrossEntropyLoss()

# Log model graph
sample_input = torch.randn(1, 3, 32, 32).to(device)
writer.add_graph(tb_model, sample_input)

print("TensorBoard writer created.")
print(f"Log directory: {log_dir}")
print("\nTraining with full TensorBoard logging...")

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(40):
    # Training
    tb_model.train()
    running_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = tb_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    tb_model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = tb_model(images)
            vl += criterion(outputs, labels).item() * images.size(0)
            _, predicted = outputs.max(1)
            vt += labels.size(0)
            vc += predicted.eq(labels).sum().item()

    val_loss = vl / vt
    val_acc = vc / vt

    # Log to TensorBoard
    writer.add_scalars('Loss', {'train': train_loss, 'val': val_loss}, epoch)
    writer.add_scalars('Accuracy', {'train': train_acc, 'val': val_acc}, epoch)
    writer.add_scalar('Learning_Rate', optimizer.param_groups[0]['lr'], epoch)

    # Log weight histograms every 5 epochs
    if epoch % 5 == 0:
        for name, param in tb_model.named_parameters():
            writer.add_histogram(f'weights/{name}', param.data, epoch)
            if param.grad is not None:
                writer.add_histogram(f'gradients/{name}', param.grad, epoch)

    scheduler.step(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = copy.deepcopy(tb_model.state_dict())
    else:
        patience_counter += 1
        if patience_counter >= 10:
            print(f"Early stopping at epoch {epoch + 1}")
            tb_model.load_state_dict(best_state)
            break

writer.close()
print(f"\nTraining complete. TensorBoard logs saved to: {log_dir}")
print("\nTo view in Colab:")
print("  %load_ext tensorboard")
print(f"  %tensorboard --logdir {log_dir}")


## 10. Optuna Hyperparameter Optimization

Optuna is the go-to hyperparameter tuning library for PyTorch (analogous to Keras Tuner for TensorFlow). It uses efficient sampling strategies including Tree-structured Parzen Estimator (TPE).


In [ ]:
# ============================================================
# 10a — Define Optuna objective
# ============================================================
import optuna
from optuna.trial import TrialState

def optuna_objective(trial):
    """Optuna objective function: build, train, and evaluate a model."""

    # Hyperparameter search space
    num_filters_1 = trial.suggest_categorical('filters_1', [16, 32, 64])
    num_filters_2 = trial.suggest_categorical('filters_2', [32, 64, 128])
    dense_units = trial.suggest_categorical('dense_units', [64, 128, 256])
    dropout_rate = trial.suggest_float('dropout', 0.0, 0.5, step=0.1)
    use_batchnorm = trial.suggest_categorical('batchnorm', [True, False])
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)

    # Build model dynamically
    model_layers = []
    # Block 1
    model_layers.append(nn.Conv2d(3, num_filters_1, 3, padding=1))
    if use_batchnorm: model_layers.append(nn.BatchNorm2d(num_filters_1))
    model_layers.append(nn.ReLU())
    model_layers.append(nn.MaxPool2d(2))
    if dropout_rate > 0: model_layers.append(nn.Dropout2d(dropout_rate))

    # Block 2
    model_layers.append(nn.Conv2d(num_filters_1, num_filters_2, 3, padding=1))
    if use_batchnorm: model_layers.append(nn.BatchNorm2d(num_filters_2))
    model_layers.append(nn.ReLU())
    model_layers.append(nn.MaxPool2d(2))
    if dropout_rate > 0: model_layers.append(nn.Dropout2d(dropout_rate))

    # Flatten + Dense
    model_layers.append(nn.Flatten())
    model_layers.append(nn.Linear(num_filters_2 * 8 * 8, dense_units))
    model_layers.append(nn.ReLU())
    if dropout_rate > 0: model_layers.append(nn.Dropout(dropout_rate))
    model_layers.append(nn.Linear(dense_units, 10))

    model = nn.Sequential(*model_layers).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    # Train for limited epochs
    for epoch in range(15):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

        # Validate
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                _, predicted = model(images).max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_acc = correct / total

        # Pruning: stop unpromising trials early
        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_acc

print("Optuna objective defined. Search space:")
print("  - Filters: [16/32/64] & [32/64/128]")
print("  - Dense units: [64, 128, 256]")
print("  - Dropout: 0.0 - 0.5")
print("  - BatchNorm: True/False")
print("  - LR: 1e-4 to 1e-2 (log)")
print("  - Weight decay: 1e-6 to 1e-3 (log)")


In [ ]:
# ============================================================
# 10b — Run Optuna study
# ============================================================
study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3)
)

print("Running Optuna hyperparameter search (20 trials)...\n")
study.optimize(optuna_objective, n_trials=20, show_progress_bar=True)

# Results
print("\n" + "=" * 60)
print("SEARCH COMPLETE")
print("=" * 60)
print(f"\nBest trial accuracy: {study.best_trial.value:.4f}")
print("\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

# Visualization
fig = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title("Optuna Optimization History", fontweight='bold')
plt.tight_layout()
plt.show()

fig = optuna.visualization.matplotlib.plot_param_importances(study)
plt.title("Hyperparameter Importance", fontweight='bold')
plt.tight_layout()
plt.show()


## 11. torchvision Data Augmentation

PyTorch's `torchvision.transforms` provides a comprehensive augmentation pipeline that integrates directly with `DataLoader`.


In [ ]:
# ============================================================
# 11a — Define augmentation pipelines
# ============================================================
# Standard augmentation
train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(32, padding=4),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToTensor(),
])

# Aggressive augmentation (AutoAugment + Cutout)
aggressive_transform = T.Compose([
    T.AutoAugment(policy=T.AutoAugmentPolicy.CIFAR10),
    T.ToTensor(),
    T.RandomErasing(p=0.3, scale=(0.02, 0.2)),  # Cutout
])

# Visualize transforms
raw_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False)

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for i in range(8):
    raw_img = raw_dataset[i][0]  # PIL Image

    axes[0, i].imshow(raw_img)
    axes[0, i].set_title("Original", fontsize=8)
    axes[0, i].axis('off')

    aug_img = train_transform(raw_img).permute(1, 2, 0).numpy()
    axes[1, i].imshow(np.clip(aug_img, 0, 1))
    axes[1, i].set_title("Standard Aug", fontsize=8)
    axes[1, i].axis('off')

    agg_img = aggressive_transform(raw_img).permute(1, 2, 0).numpy()
    axes[2, i].imshow(np.clip(agg_img, 0, 1))
    axes[2, i].set_title("AutoAugment", fontsize=8)
    axes[2, i].axis('off')

plt.suptitle("torchvision Augmentation Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 11b — A/B test: augmentation impact on accuracy
# ============================================================
# Create augmented dataset
aug_train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False, transform=train_transform)
aug_train_dataset, _ = random_split(aug_train_dataset, [40000, 10000],
                                     generator=torch.Generator().manual_seed(42))
aug_loader = DataLoader(aug_train_dataset, batch_size=128, shuffle=True, num_workers=2)

# Train with augmentation (using the augmented loader)
print("Training with torchvision augmentation pipeline...")
aug_model = FlexibleCNN(use_dropout=True, dropout_rate=0.2).to(device)
optimizer = optim.Adam(aug_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

aug_history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

for epoch in range(30):
    aug_model.train()
    rl, c, t = 0, 0, 0
    for images, labels in aug_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = aug_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        rl += loss.item() * images.size(0)
        _, pred = outputs.max(1)
        t += labels.size(0)
        c += pred.eq(labels).sum().item()

    aug_history['loss'].append(rl/t)
    aug_history['accuracy'].append(c/t)

    aug_model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = aug_model(images)
            vl += criterion(outputs, labels).item() * images.size(0)
            _, pred = outputs.max(1)
            vt += labels.size(0)
            vc += pred.eq(labels).sum().item()
    aug_history['val_loss'].append(vl/vt)
    aug_history['val_accuracy'].append(vc/vt)

plot_ab(hist_baseline, HistoryWrapper(aug_history), "No Augmentation", "torchvision Aug")


## 12. Multi-Domain Data Augmentation

Demonstrating augmentation for image, text, time series, tabular, and audio data in PyTorch.


### 12a. Image — Albumentations + torchvision

In [ ]:
# ============================================================
# 12a — Albumentations (popular PyTorch-compatible library)
# ============================================================
import albumentations as A

album_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
    A.CoarseDropout(max_holes=8, max_height=4, max_width=4, p=0.3),
    A.GaussNoise(var_limit=(10, 50), p=0.3),
    A.ElasticTransform(alpha=30, sigma=5, p=0.2),
])

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i in range(6):
    raw_img = np.array(raw_dataset[i][0])
    axes[0, i].imshow(raw_img)
    axes[0, i].set_title("Original", fontsize=8); axes[0, i].axis('off')

    aug_result = album_pipeline(image=raw_img)['image']
    axes[1, i].imshow(aug_result)
    axes[1, i].set_title("Albumentations", fontsize=8); axes[1, i].axis('off')

plt.suptitle("Albumentations Augmentation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 12b. Text Augmentation — nlpaug

In [ ]:
# ============================================================
# 12b — Text augmentation with nlpaug
# ============================================================
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.char as nac

sample_texts = [
    "The movie was absolutely fantastic and I loved every minute of it.",
    "Deep learning models need careful hyperparameter tuning to work well.",
    "The weather forecast predicts heavy rain throughout the weekend.",
]

syn_aug = naw.SynonymAug(aug_src='wordnet')
char_aug = nac.RandomCharAug(action='insert', aug_char_min=1, aug_char_max=2)
swap_aug = naw.RandomWordAug(action='swap', aug_p=0.2)

print("TEXT AUGMENTATION WITH nlpaug")
print("=" * 70)
for text in sample_texts:
    print(f"\nOriginal:  {text}")
    print(f"Synonym:   {syn_aug.augment(text)[0]}")
    print(f"Char Ins:  {char_aug.augment(text)[0]}")
    print(f"Word Swap: {swap_aug.augment(text)[0]}")
    print("-" * 70)


### 12c. Time Series Augmentation — tsaug

In [ ]:
# ============================================================
# 12c — Time series augmentation
# ============================================================
from tsaug import TimeWarp, AddNoise, Drift, Quantize, Reverse

np.random.seed(42)
t = np.linspace(0, 4 * np.pi, 200)
signal = np.sin(t) + 0.3 * np.sin(3 * t) + 0.1 * np.random.randn(len(t))
signal = signal.reshape(1, -1, 1)

augmenters = {
    "TimeWarp": TimeWarp(n_speed_change=3, max_speed_ratio=3),
    "AddNoise": AddNoise(scale=0.1),
    "Drift": Drift(max_drift=0.3),
    "Quantize": Quantize(n_levels=20),
    "Reverse": Reverse(),
}

fig, axes = plt.subplots(len(augmenters) + 1, 1, figsize=(14, 10), sharex=True)
axes[0].plot(signal[0, :, 0], color='black', linewidth=1.5)
axes[0].set_title("Original", fontweight='bold'); axes[0].grid(True, alpha=0.3)

for i, (name, aug) in enumerate(augmenters.items(), 1):
    augmented = aug.augment(signal)
    axes[i].plot(augmented[0, :, 0], linewidth=1, color=plt.cm.Set1(i/len(augmenters)))
    axes[i].set_title(name, fontweight='bold'); axes[i].grid(True, alpha=0.3)

plt.suptitle("Time Series Augmentation (tsaug)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 12d. Tabular Data Augmentation

In [ ]:
# ============================================================
# 12d — Tabular augmentation: noise injection + mixup
# ============================================================
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from collections import Counter

X_tab, y_tab = make_classification(
    n_samples=1000, n_features=20, n_informative=15,
    n_classes=3, weights=[0.7, 0.2, 0.1], random_state=42)

print("Original distribution:", Counter(y_tab))

# Noise injection for minority classes
aug_X, aug_y = [X_tab.copy()], [y_tab.copy()]
for cls in [1, 2]:
    n_need = Counter(y_tab)[0] - Counter(y_tab)[cls]
    indices = np.where(y_tab == cls)[0]
    selected = np.random.choice(indices, size=n_need, replace=True)
    noise = np.random.normal(0, 0.1, (n_need, X_tab.shape[1]))
    aug_X.append(X_tab[selected] + noise)
    aug_y.append(np.full(n_need, cls))

X_aug = np.vstack(aug_X)
y_aug = np.concatenate(aug_y)
print("After augmentation:", Counter(y_aug))

# Visualize
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tab)
X_new_pca = pca.transform(X_aug[len(X_tab):])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_tab, cmap='Set1', alpha=0.6, s=15)
axes[0].set_title("Original", fontweight='bold')

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_tab, cmap='Set1', alpha=0.3, s=15)
axes[1].scatter(X_new_pca[:, 0], X_new_pca[:, 1], c='orange', alpha=0.5, s=10, marker='x')
axes[1].set_title("After Noise Augmentation", fontweight='bold')

plt.suptitle("Tabular Data Augmentation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 12e. Audio/Speech Augmentation — audiomentations

In [ ]:
# ============================================================
# 12e — Audio augmentation
# ============================================================
from audiomentations import (
    Compose as AudioCompose, AddGaussianNoise, TimeStretch,
    PitchShift, Shift, Gain
)

sr = 16000
duration = 2.0
t_audio = np.linspace(0, duration, int(sr * duration), dtype=np.float32)
audio = (0.5 * np.sin(2 * np.pi * 440 * t_audio) +
         0.3 * np.sin(2 * np.pi * 880 * t_audio) +
         0.1 * np.random.randn(len(t_audio)).astype(np.float32))

audio_aug = AudioCompose([
    AddGaussianNoise(min_amplitude=0.005, max_amplitude=0.02, p=0.8),
    TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.5),
    Shift(min_shift=-0.2, max_shift=0.2, p=0.5),
    Gain(min_gain_db=-6, max_gain_db=6, p=0.5),
])

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
axes[0].plot(t_audio[:3000], audio[:3000], color='black', linewidth=0.5)
axes[0].set_title("Original Audio", fontweight='bold'); axes[0].grid(True, alpha=0.3)

for i in range(1, 4):
    aug_audio = audio_aug(samples=audio, sample_rate=sr)
    axes[i].plot(t_audio[:3000], aug_audio[:3000], linewidth=0.5,
                 color=plt.cm.Set2(i/4))
    axes[i].set_title(f"Augmented v{i}", fontweight='bold'); axes[i].grid(True, alpha=0.3)

plt.suptitle("Audio Data Augmentation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

| Technique | PyTorch Implementation | Key Insight |
|---|---|---|
| **L1 Regularization** | Manual penalty in loss | Creates sparse weights |
| **L2 Regularization** | `weight_decay` in optimizer | Shrinks all weights uniformly |
| **Dropout** | `nn.Dropout` | Forces redundant learning |
| **Early Stopping** | Custom callback class | Free lunch — always use it |
| **MC Dropout** | `model.train()` at inference | Uncertainty estimation |
| **He Initialization** | `nn.init.kaiming_normal_` | Default for ReLU networks |
| **Batch Normalization** | `nn.BatchNorm2d` / `nn.BatchNorm1d` | Enables higher LR |
| **Custom Regularizers** | Add penalty to loss | Flexible, task-specific |
| **TensorBoard** | `SummaryWriter` | Interactive training visualization |
| **Optuna** | `study.optimize()` | Automated HP search with pruning |
| **Data Augmentation** | `torchvision.transforms` + Albumentations | Larger effective dataset |

---
*Notebook generated for Assignment Part 1 — PyTorch Edition*
